# [MODEL] 현지 최종 책임

EfficientNet 전이학습 기반 식재료 이미지 분류 학습 노트북이다.
클래스 목록, train/validation/test 분할, 데이터 증강 설정에 이어
백본 동결 후 1~3 epoch 빠른 학습, 학습 곡선 시각화, test 평가(정확도/혼동행렬),
모델·클래스명 저장, 예측 데모 함수(predict_image)까지 전체 파이프라인을 완성한다.

In [ ]:
# 재현성과 데이터 처리, 시각화, 평가에 필요한 라이브러리를 임포트한다
import os
import json
import random

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# 재현성을 위한 시드 고정
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 1. 클래스 목록 확정

외형이 서로 비슷해 혼동되기 쉬운 클래스(예: 양파-마늘, 애호박-오이, 청양고추-오이고추 등)는
제외하고, 형태·색상 구분이 뚜렷한 6개 클래스만 선정한다.

In [ ]:
# 데이터 경로 (실제 데이터셋 확보 후 사용자 환경에 맞게 수정)
RAW_DATA_DIR = './data/raw'
SPLIT_DATA_DIR = './data/split'

# 외형이 뚜렷하게 구분되는 6개 클래스만 선정한다 (5~8개 범위 내)
CLASS_NAMES = ['potato', 'carrot', 'cabbage', 'tomato', 'eggplant', 'paprika']

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

print('선정된 클래스:', CLASS_NAMES)

## 2. train / validation / test 8:1:1 분할

클래스별 폴더 구조(`RAW_DATA_DIR/클래스명/*.jpg`)를 가정하고,
각 클래스 내 이미지를 8:1:1 비율로 분리하여 `SPLIT_DATA_DIR` 하위에 복사한다.

In [ ]:
import shutil


def split_dataset(raw_dir, split_dir, class_names, ratios=(0.8, 0.1, 0.1), seed=SEED):
    """클래스별 이미지를 train/val/test 폴더로 8:1:1 비율로 복사한다"""
    rng = random.Random(seed)
    for split in ['train', 'val', 'test']:
        for cls in class_names:
            os.makedirs(os.path.join(split_dir, split, cls), exist_ok=True)

    for cls in class_names:
        cls_dir = os.path.join(raw_dir, cls)
        files = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        rng.shuffle(files)

        n_total = len(files)
        n_train = int(n_total * ratios[0])
        n_val = int(n_total * ratios[1])

        splits = {
            'train': files[:n_train],
            'val': files[n_train:n_train + n_val],
            'test': files[n_train + n_val:],
        }

        for split, split_files in splits.items():
            for fname in split_files:
                src = os.path.join(cls_dir, fname)
                dst = os.path.join(split_dir, split, cls, fname)
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)

        print(f'{cls}: train={len(splits["train"])}, val={len(splits["val"])}, test={len(splits["test"])}')


# 실제 분할 실행은 아래 4번 셀(데이터셋 로딩)에서 함께 수행한다

## 3. ImageDataGenerator 증강 설정 (참고용)

`ImageDataGenerator`/`flow_from_directory` 방식의 증강 설정을 참고용으로 정의해 둔다.
실제 학습 파이프라인은 5번 항목에서 `tf.data.Dataset` 기반의 증강 레이어를 사용한다.

In [ ]:
# train에만 적용할 증강: 회전, 좌우 반전, 밝기 조절
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    horizontal_flip=True,
    brightness_range=(0.8, 1.2),
)

# validation/test에는 증강을 적용하지 않고 스케일링만 수행한다
val_test_datagen = ImageDataGenerator(rescale=1.0 / 255)


## 4. train / validation / test 데이터셋 로딩

`SPLIT_DATA_DIR`에 `split_dataset`으로 분리해 둔 이미지를 `image_dataset_from_directory`로 불러온다.
클래스 순서를 `CLASS_NAMES`로 고정해 이후 저장할 `class_names.json`과 예측 결과가 항상 같은 순서를 갖도록 한다.

In [ ]:
# 이번 주 확보한 데이터셋을 8:1:1로 분리한다 (이미 분리되어 있다면 재실행 불필요)
split_dataset(RAW_DATA_DIR, SPLIT_DATA_DIR, CLASS_NAMES)

train_dir = os.path.join(SPLIT_DATA_DIR, 'train')
val_dir = os.path.join(SPLIT_DATA_DIR, 'val')
test_dir = os.path.join(SPLIT_DATA_DIR, 'test')

# 클래스 순서를 고정하기 위해 class_names를 명시적으로 지정한다
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, labels='inferred', label_mode='categorical',
    class_names=CLASS_NAMES, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir, labels='inferred', label_mode='categorical',
    class_names=CLASS_NAMES, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir, labels='inferred', label_mode='categorical',
    class_names=CLASS_NAMES, image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False)

## 5. 데이터 증강 레이어 적용 및 전처리 완성

앞서 정의한 `train_datagen`/`val_test_datagen`(ImageDataGenerator)은 `flow_from_directory` 전용이므로,
`image_dataset_from_directory`로 불러온 `tf.data.Dataset`에는 별도의 증강 레이어를 적용한다.
회전·좌우 반전·밝기 조절을 train에만 적용하고, val/test는 원본을 그대로 사용해 데이터 전처리를 마무리한다.

In [ ]:
# 회전, 좌우 반전, 밝기 조절을 적용하는 증강 레이어 (train에만 적용)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomBrightness(0.2),
], name='data_augmentation')

AUTOTUNE = tf.data.AUTOTUNE

# train 데이터셋에는 증강을 적용하고, val/test는 원본 그대로 사용한다
train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

## 6. EfficientNetB0 백본 로드 및 분류기 헤드 구성

In [ ]:
# ImageNet 사전학습 가중치를 사용하는 EfficientNetB0 백본 (최상단 분류층 제외)
base_model = tf.keras.applications.EfficientNetB0(
    weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,))

# 백본을 동결하여 새로 추가한 분류기 헤드만 빠르게 학습시킨다
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
# EfficientNet 전용 전처리(0~255 -> 모델이 기대하는 범위)를 적용한다
x = tf.keras.applications.efficientnet.preprocess_input(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'])

model.summary()

## 7. 백본 동결 상태로 빠른 학습 (1~3 epoch)

백본은 동결한 채 분류기 헤드만 짧게 학습시켜, 전체 파이프라인이 정상 동작하는지와
초기 성능 수준을 빠르게 확인한다.

In [ ]:
# 백본을 동결한 채 분류기 헤드만 1~3 epoch만 짧게 학습시켜 빠르게 성능을 확인한다
EPOCHS = 3

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS)

## 8. 학습 곡선 시각화

In [ ]:
# 학습/검증 정확도와 손실을 그래프로 시각화하여 학습 추이를 확인한다
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(1, len(acc) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, acc, label='train accuracy')
axes[0].plot(epochs_range, val_acc, label='val accuracy')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs_range, loss, label='train loss')
axes[1].plot(epochs_range, val_loss, label='val loss')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. 모델 평가: test accuracy 및 혼동행렬

In [ ]:
# test 데이터셋으로 최종 정확도를 평가한다
test_loss, test_accuracy = model.evaluate(test_ds)
print(f'Test accuracy: {test_accuracy:.4f}')

In [ ]:
# 실제 라벨과 예측 라벨을 모아 혼동행렬을 계산한다
y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)

fig, ax = plt.subplots(figsize=(7, 7))
disp.plot(ax=ax, xticks_rotation=45, cmap='Blues', colorbar=False)
plt.title('Confusion Matrix (Test Set)')
plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

## 10. 모델 및 클래스명 저장

In [ ]:
# 학습된 모델과 클래스명 순서를 model/artifacts 하위에 저장한다
ARTIFACT_DIR = './model/artifacts'
os.makedirs(ARTIFACT_DIR, exist_ok=True)

MODEL_PATH = os.path.join(ARTIFACT_DIR, 'ingredient_model.keras')
CLASS_NAMES_PATH = os.path.join(ARTIFACT_DIR, 'class_names.json')

model.save(MODEL_PATH)

# 클래스명은 저장 시 예측 결과의 인덱스와 항상 같은 순서를 유지해야 하므로 CLASS_NAMES를 그대로 기록한다
with open(CLASS_NAMES_PATH, 'w', encoding='utf-8') as f:
    json.dump(CLASS_NAMES, f, ensure_ascii=False, indent=2)

print(f'모델 저장 완료: {MODEL_PATH}')
print(f'클래스명 저장 완료: {CLASS_NAMES_PATH}')

## 11. 예측 함수 데모 (predict_image)

In [ ]:
def predict_image(path, model=model, class_names=CLASS_NAMES, img_size=IMG_SIZE):
    """이미지 경로를 받아 예측 클래스명, 확신도, top3 결과를 딕셔너리로 반환한다"""
    img = tf.keras.utils.load_img(path, target_size=img_size)
    img_array = tf.keras.utils.img_to_array(img)
    img_array = tf.expand_dims(img_array, axis=0)

    predictions = model.predict(img_array, verbose=0)[0]

    top3_idx = np.argsort(predictions)[::-1][:3]
    top3 = [
        {'name': class_names[i], 'confidence': float(predictions[i])}
        for i in top3_idx
    ]

    best_idx = int(top3_idx[0])
    return {
        'name': class_names[best_idx],
        'confidence': float(predictions[best_idx]),
        'top3': top3,
    }

In [ ]:
# 테스트 이미지 한 장으로 예측 함수 데모를 실행한다
sample_class = CLASS_NAMES[0]
sample_dir = os.path.join(test_dir, sample_class)
sample_image_path = os.path.join(sample_dir, os.listdir(sample_dir)[0])

result = predict_image(sample_image_path)
print('예측 결과:', result)

# 예측에 사용한 이미지를 함께 시각화한다
img = tf.keras.utils.load_img(sample_image_path, target_size=IMG_SIZE)
plt.imshow(img)
plt.title(f"pred: {result['name']} ({result['confidence']:.2%})")
plt.axis('off')
plt.show()